In [1]:
# Import basic libraries first (NOT DuckDuckGoSearchRun yet)
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import  tool
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai
from langchain_google_vertexai import ChatVertexAI

import requests
import random
import urllib3
import ssl
import os
import certifi

# CRITICAL: Disable SSL verification BEFORE loading environment or making any API calls
os.environ["LANGCHAIN_SSL_VERIFY"] = "false"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["SSL_CERT_FILE"] = ""
os.environ["REQUESTS_VERIFY_SSL"] = "0"
os.environ["PYTHONHTTPSVERIFY"] = "0"

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Override SSL context
ssl._create_default_https_context = ssl._create_unverified_context

from dotenv import load_dotenv
load_dotenv()

# Monkey-patch requests to disable SSL verification
try:
    from functools import partialmethod
    if not hasattr(requests.Session, '_ssl_bypass_applied'):
        requests.Session.request = partialmethod(requests.Session.request, verify=False)
        requests.request = partialmethod(requests.request, verify=False)
        requests.Session._ssl_bypass_applied = True
        print("✓ requests SSL verification disabled")
    else:
        print("✓ requests already patched")
except Exception as e:
    print(f"Warning: Could not patch requests library: {e}")

# Patch httpx client (DDGS uses httpx)
try:
    import httpx
    if not hasattr(httpx.Client, '_ssl_bypass_applied'):
        _original_httpx_client_init = httpx.Client.__init__
        _original_httpx_async_client_init = httpx.AsyncClient.__init__
        
        def patched_httpx_client_init(self, *args, **kwargs):
            kwargs['verify'] = False
            _original_httpx_client_init(self, *args, **kwargs)
        
        def patched_httpx_async_client_init(self, *args, **kwargs):
            kwargs['verify'] = False
            _original_httpx_async_client_init(self, *args, **kwargs)
        
        httpx.Client.__init__ = patched_httpx_client_init
        httpx.AsyncClient.__init__ = patched_httpx_async_client_init
        httpx.Client._ssl_bypass_applied = True
        print("✓ httpx SSL verification disabled")
    else:
        print("✓ httpx already patched")
except Exception as e:
    print(f"Warning: Could not patch httpx: {e}")

# Patch aiohttp if present
try:
    import aiohttp
    if not hasattr(aiohttp.ClientSession, '_ssl_bypass_applied'):
        _original_aiohttp_init = aiohttp.ClientSession.__init__
        
        def patched_aiohttp_init(self, *args, **kwargs):
            kwargs['connector'] = aiohttp.TCPConnector(ssl=False)
            _original_aiohttp_init(self, *args, **kwargs)
        
        aiohttp.ClientSession.__init__ = patched_aiohttp_init
        aiohttp.ClientSession._ssl_bypass_applied = True
        print("✓ aiohttp SSL verification disabled")
    else:
        print("✓ aiohttp already patched")
except Exception as e:
    print(f"Info: aiohttp not available or already patched: {e}")

# NOW import DuckDuckGoSearchRun AFTER all SSL patches are in place
from langchain_community.tools import DuckDuckGoSearchRun
print("✓ DuckDuckGoSearchRun imported after SSL patches")


✓ requests SSL verification disabled
✓ httpx SSL verification disabled
✓ aiohttp SSL verification disabled
✓ DuckDuckGoSearchRun imported after SSL patches


### Defining the tool

In [2]:
# from langchain_community.tools import WikipediaQueryRun
# from langchain_community.utilities import WikipediaAPIWrapper

# # Wikipedia doesn't have SSL issues in most corporate networks
# search_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [3]:

# Suppress browser impersonation warnings by using basic mode
import warnings
warnings.filterwarnings('ignore', message='Impersonate.*does not exist')

search_tool = DuckDuckGoSearchRun()


In [4]:
# Test the DuckDuckGo search tool
# result = search_tool.invoke("Python programming language")
# print(result)

In [5]:

@tool
def calculator(first_number: float, second_number: float, operation: str):
    """Performs basic arithmetic operations on two numbers.
    Args:
        first_number (float): The first number.
        second_number (float): The second number.
        operation (str): The operation to perform. Can be 'add', 'subtract', 'multiply', or 'divide'.
    """
    try :
        if operation == 'add':
            result =  first_number + second_number
        elif operation == 'subtract':
            result = first_number - second_number
        elif operation == 'multiply':
            result =    first_number * second_number
        elif operation == 'divide':
            if second_number == 0:
                raise ValueError("Cannot divide by zero")
            result = first_number / second_number
        else:
            return {"error ":f"unsupported operation '{operation}'. "}
        return result
    except Exception as e:
        return {"error": str(e)}

In [6]:
@tool
def get_stock_price(stock_symbol: str):
    """Fetches the current stock price for a given stock symbol.
    Args:
        stock_symbol (str): The stock symbol to fetch the price for.
    Returns:
        dict: A dictionary containing the stock symbol and its current price.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={stock_symbol}&apikey={os.getenv('ALPHA_VANTAGE_API_KEY')}"

    # Disable SSL verification for corporate network
    r = requests.get(url, verify=False)
    data = r.json()

    return data 

In [7]:
# # Test the tool by invoking it correctly
# print(get_stock_price.invoke({"stock_symbol": "AAPL"}))

In [8]:
# Step 1: Create a list of all your tools
tools = [search_tool, calculator, get_stock_price]

# Step 2: Create your LLM instance
llm = ChatVertexAI(
    model_name="gemini-2.0-flash",
    project=os.getenv('GEMINI_API_KEY'),
    location="us-central1",
    temperature=0.3,  # Lower temperature for more focused responses
    max_output_tokens=50,  # Limit response length (adjust as needed: 50-200)
)

# Step 3: Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

C:\Users\H604660\AppData\Local\Temp\ipykernel_16820\2505615384.py:5: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(
C:\Users\H604660\AppData\Local\Temp\ipykernel_16820\2505615384.py:5: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(


In [9]:
# Alternative configuration options:
# - You can also try model_name="gemini-1.5-pro-002" or "gemini-1.5-flash-002"
# - Adjust temperature between 0 (deterministic) and 1 (creative)
# - Change location to your preferred region (e.g., "us-east1", "europe-west1")

In [10]:
#state
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [11]:
def chat_node(state:ChatState):
    """ LLM node that may answer or request a tool call"""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages":[response]}


tool_node = ToolNode(tools)

In [14]:
#Build the graph
graph = StateGraph(ChatState)

# Add nodes
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)  # Must be named 'tools' for tools_condition compatibility

# Add edges
graph.add_edge(START, 'chat_node')

# If the LLM asked for a tool call, go to the tools node, else end the execution
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools','chat_node')


# Compile the graph
chatbot = graph.compile()

In [15]:
#Displaying the graph
print(chatbot.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +-----------+         
        | chat_node |         
        +-----------+         
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [16]:

content = "Score by Samson in recent india vs West Indies match" 
out = chatbot.invoke({"messages":[HumanMessage(content=content  )]})

print(out["messages"][-1].content)

c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\myenv\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


DDGSException: ConnectError: ConnectError('error sending request for url (https://search.brave.com/search?q=Samson+score+in+recent+India+vs+West+Indies+match&source=web&tf=py)', 'https://search.brave.com/search?q=Samson+score+in+recent+India+vs+West+Indies+match&source=web&tf=py')